# 03 · Working with GEOtop from Python

What usually needs external scripts around the C++ executable:

1. reading parameters as the model sees them;
2. parametric runs, in parallel;
3. diagnostics of every internal step;
4. calling the model's laws as functions.

Runs with modified parameters are no longer validated against the C++ model.

In [ ]:
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
from _helpers import RUNS, point_df, prepare_case, set_keywords

from geotop_py import run_simulation

PY = sys.executable  # the interpreter GEOtopPY is installed in

## 1. Parameters as the model reads them

`parfile.parse` reads `geotop.inpts` with GEOtop's own keyword tables.
`number(name, component, default)` returns the declared value or the default,
as GEOtop's `assignation_number` does; an explicit `-9999` counts as absent.

In [ ]:
from geotop_py.io import parfile

NAME = "Matsch_P2_Ref_007"
base = prepare_case(NAME)
pf = parfile.parse(str(base / "geotop.inpts"))

for key in ["TimeStepEnergyAndWater", "SnowCorrFactor", "ThresTempSnow", "ThresTempRain"]:
    declared = pf.has_number(key)
    print(f"{key:24s} {pf.number(key):>10g}  {'declared' if declared else 'not declared'}")

A misspelled keyword is an error in Python, rather than a parameter that
silently takes its default:

In [ ]:
try:
    pf.number("SnowCorrectionFactor")
except KeyError as e:
    print("KeyError:", e)

## 2. Parametric runs

Each variant is a copy of the case with one keyword changed by
`set_keywords`, run as its own `GEOtopPY` process, as you would launch several
`geotop` runs. The threads only wait for the processes, so the runs go in
parallel on the available cores. Here: the snowfall correction factor
(`SnowCorrFactor`, 1.398 in the reference case).

In [ ]:
import os
import subprocess
from concurrent.futures import ThreadPoolExecutor


def run_variant(factor):
    case = prepare_case(NAME, RUNS / f"Matsch_P2_SnowCorrFactor_{factor}")
    set_keywords(case, SnowCorrFactor=factor)
    subprocess.run([PY, "-m", "geotop_py", str(case), "-q"], check=True)
    return case


factors = [1.0, 1.398, 2.0, 3.0, 4.0]
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    dirs = list(pool.map(run_variant, factors))
print(f"{len(factors)} runs in {time.perf_counter() - t0:.1f} s")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
window = slice("2009-11-01", "2009-11-02 12:00")   # the snow event
for f, d in zip(factors, dirs):
    df = point_df(d / "output-tabs" / "point0001.txt").loc[window]
    style = dict(lw=2.5, color="k") if f == 1.398 else dict(lw=1)
    axes[0].plot(df.index, df["snow_depth[mm]"], label=f"SnowCorrFactor = {f}", **style)
    axes[1].plot(df.index, df["snow_water_equivalent[mm]"], **style)
axes[0].set_ylabel("snow_depth [mm]")
axes[1].set_ylabel("SWE [mm]")
axes[0].legend(fontsize=8)
axes[0].set_title("black: the reference configuration", fontsize=9)
fig.tight_layout()

## 3. Internal steps

The records returned by `run_simulation` cover every internal step,
including the halved ones, with the energy and Richards solver diagnostics.
With the C++ model this information is only in the log.

In [ ]:
from geotop_py import steps_table

results = run_simulation(str(base), verbose=False)
steps = pd.DataFrame(steps_table(results[1])).set_index("date")
print(len(steps), "internal steps")
print("step lengths [s]:", steps["dt"].value_counts().sort_index().to_dict())
print("steps that needed more than one trial:", int((steps["trials"] > 1).sum()))
print("non-converged energy / water steps:",
      int((~steps["converged"]).sum()), "/", int((~steps["wb_converged"]).sum()))

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
axes[0].plot(steps.index, steps["solver_iterations"], lw=0.7)
axes[0].set_ylabel("energy Newton\niterations")
axes[1].plot(steps.index, steps["wb_iterations"], lw=0.7)
axes[1].set_ylabel("Richards\niterations")
axes[2].plot(steps.index, steps["wb_loss"].cumsum(), lw=1)
axes[2].set_ylabel("cumulative Richards\nmass error [mm]")
axes[3].plot(steps.index, steps["depth"], lw=1, label="snow depth [mm]")
axes[3].plot(steps.index, steps["meteo_Ta"], lw=0.7, label="air temperature [C]")
axes[3].axhline(0, color="k", lw=0.5)
axes[3].legend(fontsize=8)
fig.tight_layout()

Where does the energy solver work hardest? The steps with the most iterations:

In [ ]:
steps.nlargest(8, "solver_iterations")[
    ["dt", "trials", "solver_iterations", "Tg", "depth", "meteo_Ta", "meteo_Psnow", "diag_SWnet"]]

## 4. Laws as functions

The constitutive laws are plain functions in `geotop_py.laws`, with the
arguments and units of the C++ (pressure in mm, alpha in 1/mm). Here, the
van Genuchten retention curve of the Matsch soil, read with the model's own
soil loader (file values plus keyword defaults).

In [ ]:
import numpy as np

from geotop_py import constants as C
from geotop_py import laws
from geotop_py.io import soil as io_soil

soil = io_soil.load(pf, str(base))
l = 1   # top layer
s, r = soil.pa[C.jsat][l], soil.pa[C.jres][l]
a, n = soil.pa[C.ja][l], soil.pa[C.jns][l]
Ss = soil.pa[C.jss][l]
print(f"theta_s={s}, theta_r={r}, alpha={a} 1/mm, n={n}, Ss={Ss} 1/mm")

psi = -np.logspace(0, 6, 200)     # mm
theta = [laws.teta_psi(p, 0.0, s, r, a, n, 1 - 1 / n, C.PsiMin, Ss) for p in psi]
theta_frozen = [laws.teta_psi(p, 0.1, s, r, a, n, 1 - 1 / n, C.PsiMin, Ss) for p in psi]

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(-psi, theta, label="no ice")
ax.semilogx(-psi, theta_frozen, label="ice content 0.1")
ax.set_xlabel("suction -psi [mm]")
ax.set_ylabel("liquid water content [-]")
ax.legend();

In [ ]:
# Fresh snow density (Jordan et al. 1999) as GEOtop computes it [kg/m3]
T = np.linspace(-20, 2, 100)
fig, ax = plt.subplots(figsize=(6, 4))
for u in [0, 5, 10]:
    ax.plot(T, [laws.rho_newlyfallensnow(u, t) for t in T], label=f"wind {u} m/s")
ax.set_xlabel("air temperature [C]")
ax.set_ylabel("fresh snow density [kg/m3]")
ax.legend();